In [3]:
import cv2
import numpy as np
scale_percent = 10
CONTRAST_THRESHOLD = 0

gaussians = []
gaussians_difference = []
keypoints = []

image = cv2.imread('Datasets/dataset_flowerpot-master/full_dataset/P81019-151014.jpg')
image_grey = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
image_resized = cv2.resize(image_grey, (int(image_grey.shape[1] * scale_percent / 100), int(image_grey.shape[0] * scale_percent / 100)), interpolation=cv2.INTER_AREA)    
print("Started")

def round_to_odd(x):
    return int(x) + 1 if int(x) % 2 == 0 else int(x)

def generate_gaussians(image_in=image_resized, sigma=1.6, s=3):
    k = 2 ** (1/s)
    for i in range(s + 3):  # s intervals => s+3 Gaussian images
        sigma_i = sigma * (k ** i)
        ksize = round_to_odd(6 * sigma_i)
        blur = cv2.GaussianBlur(image_in, (ksize, ksize), sigma_i)
        gaussians.append(blur)
        #gaussians.append(blur.astype(np.float32))
    

def compute_difference(listt):
    for i in range(0, len(listt)-1):
        img1 = listt[i].astype(np.float32)
        img2 = listt[i + 1].astype(np.float32)
        diff = img2 - img1
        gaussians_difference.append(diff)

def display_gaussians():
    for i in range(0, len(gaussians)):
        #cv2.imshow("Hi"+str(i), cv2.resize(gaussians[i], (int(gaussians[2].shape[1] * scale_percent / 100), int(gaussians[i].shape[0] * scale_percent / 100)), interpolation=cv2.INTER_AREA))
        cv2.imshow("Hi"+str(i), gaussians[i])

def compute_keypoints(dog_images = gaussians_difference):
    for i in range(1, len(dog_images) - 1):  # skip first and last DoG
        prev = dog_images[i - 1]
        curr = dog_images[i]
        next = dog_images[i + 1]
        h, w = curr.shape

        for y in range(1, h - 1):
            for x in range(1, w - 1):
                val = curr[y, x]
                
                # Get 3x3x3 neighborhood
                cube = np.stack([
                    prev[y-1:y+2, x-1:x+2],
                    curr[y-1:y+2, x-1:x+2],
                    next[y-1:y+2, x-1:x+2]
                ])
                
                center = cube[1,1,1]
                neighbors = np.delete(cube.flatten(), 13)  # 13 is center (3x3x3 cube)

                if abs(center) > CONTRAST_THRESHOLD:
                    if center > np.max(neighbors) or center < np.min(neighbors):

                        Dxx = curr[y, x+1] + curr[y, x-1] - 2 * curr[y, x]
                        Dyy = curr[y+1, x] + curr[y-1, x] - 2 * curr[y, x]
                        Dxy = ((curr[y+1, x+1] - curr[y+1, x-1]) - (curr[y-1, x+1] - curr[y-1, x-1])) * 0.25
                        Tr = Dxx + Dyy
                        Det = Dxx * Dyy - Dxy**2
                        
                        if Det > 0 and (Tr**2 / Det) < ((10 + 1)**2 / 10):        
                            keypoints.append((x, y, i))  # (x, y, Gaussian index)
                        
def assign_orientation(image, keypoints, patch_size=16, num_bins=36):
    keypoints_oriented = []
    for (x, y, scale_idx) in keypoints:
        # Skip keypoints too close to border
        if x < 8 or y < 8 or x >= image.shape[1] - 8 or y >= image.shape[0] - 8:
            continue
        
        patch = image[y - patch_size//2:y + patch_size//2, x - patch_size//2:x + patch_size//2]
        if patch.shape != (patch_size, patch_size):  # safety check
            continue

        # Compute gradients
        dx = cv2.Sobel(patch, cv2.CV_32F, 1, 0, ksize=3)
        dy = cv2.Sobel(patch, cv2.CV_32F, 0, 1, ksize=3)
        magnitude = np.sqrt(dx**2 + dy**2)
        orientation = (np.arctan2(dy, dx) * 180 / np.pi) % 360

        # Orientation histogram
        hist = np.zeros(num_bins)
        bin_width = 360 // num_bins

        for i in range(patch_size):
            for j in range(patch_size):
                bin_idx = int(orientation[i, j] // bin_width) % num_bins
                hist[bin_idx] += magnitude[i, j]

        # Find peak
        max_bin = np.argmax(hist)
        peak_angle = max_bin * bin_width

        keypoints_oriented.append((x, y, scale_idx, peak_angle))

    return keypoints_oriented

def draw_oriented_keypoints(image, keypoints):
    out = cv2.cvtColor(image.copy(), cv2.COLOR_GRAY2BGR)
    for (x, y, _, angle) in keypoints:
        length = 5
        angle_rad = np.deg2rad(angle)
        x2 = int(x + length * np.cos(angle_rad))
        y2 = int(y + length * np.sin(angle_rad))
        cv2.arrowedLine(out, (x, y), (x2, y2), (0, 255, 0), 1, tipLength=0.3)
    return out


def draw_keypoints(image, keypoints):
    img_with_kp = cv2.cvtColor(image.copy(), cv2.COLOR_GRAY2BGR)
    for (x, y, _) in keypoints:
        cv2.circle(img_with_kp, (x, y), 2, (0, 255, 0), 1)
    return img_with_kp

def generate_descriptors(image, keypoints, patch_size=16, num_bins=8):
    descriptors = []
    half = patch_size // 2

    for (x, y, _, angle_deg) in keypoints:
        x, y = int(x), int(y)

        # Skip keypoints near the edge
        if x < half or y < half or x >= image.shape[1] - half or y >= image.shape[0] - half:
            continue

        # Extract patch and rotate
        M = cv2.getRotationMatrix2D((x, y), -angle_deg, 1)
        rotated = cv2.warpAffine(image, M, (image.shape[1], image.shape[0]))
        patch = rotated[y - half:y + half, x - half:x + half]
        if patch.shape != (patch_size, patch_size):
            continue

        # Compute gradients
        dx = cv2.Sobel(patch, cv2.CV_32F, 1, 0, ksize=3)
        dy = cv2.Sobel(patch, cv2.CV_32F, 0, 1, ksize=3)
        magnitude = np.sqrt(dx**2 + dy**2)
        orientation = (np.arctan2(dy, dx) * 180 / np.pi) % 360

        descriptor = []

        # Divide into 4×4 subregions
        for i in range(0, patch_size, 4):
            for j in range(0, patch_size, 4):
                hist = np.zeros(num_bins)
                for u in range(4):
                    for v in range(4):
                        bin_idx = int(orientation[i+u, j+v] // (360 / num_bins)) % num_bins
                        hist[bin_idx] += magnitude[i+u, j+v]
                descriptor.extend(hist)

        # Normalize the descriptor
        descriptor = np.array(descriptor)
        descriptor /= (np.linalg.norm(descriptor) + 1e-7)
        descriptors.append(descriptor)

    return descriptors


generate_gaussians()
compute_difference(gaussians)
display_gaussians()
compute_keypoints()

img_with_kp = draw_keypoints(image_resized, keypoints)
cv2.imshow("Keypoints", img_with_kp)
print("Total Keypoints: ", len(keypoints))

oriented_kps = assign_orientation(image_resized.astype(np.float32), keypoints)
oriented_img = draw_oriented_keypoints(image_resized, oriented_kps)
cv2.imshow("Oriented Keypoints", oriented_img)
print("Keypoints with orientation:", len(oriented_kps))

desc = generate_descriptors(image_resized.astype(np.float32), oriented_kps)
print("Generated descriptors:", len(desc))
print("Each descriptor length:", len(desc[0]) if desc else 0)

cv2.waitKey(0)
cv2.destroyAllWindows()

Started
Total Keypoints:  464
Keypoints with orientation: 444
Generated descriptors: 444
Each descriptor length: 128
